In [2]:
import gensim
import gensim.downloader as api
from gensim.models import KeyedVectors
from scipy.stats import spearmanr
import nltk
import string
from nltk.corpus import gutenberg
from nltk.corpus import stopwords
from gensim.models import Word2Vec

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\LiuYe\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LiuYe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LiuYe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
model: KeyedVectors = api.load("word2vec-google-news-300")
vocab_size = len(model.key_to_index)
vector_dim = model.vector_size
vocab_size, vector_dim

(3000000, 300)

In [4]:
test_words = ['car', 'beautiful', 'play', 'computer']
for word in test_words:
    similar = model.most_similar(word, topn=10)
    for word, score in similar:
        print(f"{word}: {score:.4f}")
    print('='*50)

vehicle: 0.7821
cars: 0.7424
SUV: 0.7161
minivan: 0.6907
truck: 0.6736
Car: 0.6678
Ford_Focus: 0.6673
Honda_Civic: 0.6627
Jeep: 0.6511
pickup_truck: 0.6441
gorgeous: 0.8353
lovely: 0.8107
stunningly_beautiful: 0.7329
breathtakingly_beautiful: 0.7231
wonderful: 0.6854
fabulous: 0.6700
loveliest: 0.6613
prettiest: 0.6595
beatiful: 0.6593
magnificent: 0.6591
playing: 0.7668
plays: 0.7108
played: 0.6962
game: 0.6501
toplay: 0.5970
Playing: 0.5813
games: 0.5265
paly: 0.5261
score: 0.5229
Play: 0.5014
computers: 0.7979
laptop: 0.6640
laptop_computer: 0.6549
Computer: 0.6473
com_puter: 0.6082
technician_Leonard_Luchko: 0.5663
mainframes_minicomputers: 0.5618
laptop_computers: 0.5585
PC: 0.5540
maker_Dell_DELL.O: 0.5519


In [5]:
similarity_pairs = [
    ("car", "car", 9),
    ("computer", "laptop", 8),
    ("sun", "moon", 6),
    ("cat", "dog", 7),
    ("joy", "happiness", 8),
    ("house", "building", 7),
    ("fast", "fast", 8),
    ("red", "scarlet", 9),
    ("good", "bad", 4)
]
cosine_similarities = []
human_scores = []
for w1, w2, score in similarity_pairs:
    cos_sim = model.similarity(w1, w2)
    cosine_similarities.append(cos_sim)
    human_scores.append(score)
    print(f"{w1} - {w2} : cos_sim = {cos_sim:.4f}, human_scores = {score}")
res = spearmanr(cosine_similarities, human_scores)
print(f"\nSpearman: {res.statistic:.4f}, p_value: {res.pvalue:.4f}")

car - car : cos_sim = 1.0000, human_scores = 9
computer - laptop : cos_sim = 0.6640, human_scores = 8
sun - moon : cos_sim = 0.4263, human_scores = 6
cat - dog : cos_sim = 0.7609, human_scores = 7
joy - happiness : cos_sim = 0.6183, human_scores = 8
house - building : cos_sim = 0.4379, human_scores = 7
fast - fast : cos_sim = 1.0000, human_scores = 8
red - scarlet : cos_sim = 0.5054, human_scores = 9
good - bad : cos_sim = 0.7190, human_scores = 4

Spearman: 0.2962, p_value: 0.4390


#### Conclusion(1.3):
- A ρ value of 0.2962 is very close to zero, indicating almost no monotonic relationship between the model’s cosine similarity scores and human similarity scores.
In other words, the embedding similarities are poorly aligned with human judgments for this small set of word pairs.
- p-value = 0.4390:
The p-value tests the null hypothesis that there is no correlation.
A p-value this high (much greater than 0.05) means we cannot reject the null hypothesis, reinforcing that the observed correlation is not statistically significant.
Some identical or very similar words like "car-car" and "red-scarlet" might have high cosine similarity, but other pairs like "good-bad" (antonyms) have unexpectedly high or low similarity that doesn’t match human judgment.
With only 9 word pairs, the dataset is very small, so the Spearman correlation is also sensitive to a few mismatched pairs.

In [6]:
result = model.most_similar(positive=['king', 'queen'],
                            negative=['king'])

for word, sim in result:
    print(f"{word:<15} {sim:.4f}")

queens          0.7399
princess        0.7071
monarch         0.6384
very_pampered_McElhatton 0.6357
Queen           0.6163
NYC_anglophiles_aflutter 0.6061
Queen_Consort   0.5924
princesses      0.5908
royal           0.5637
Makobo_Modjadji 0.5570


In [7]:
def test_identity_analogy(a, b):
    result = model.most_similar(
        positive=[a, b], negative=[a], topn=1)
    print(
        f"{a:} - {a:} + {b:} === {result[0][0]:} (sim: {result[0][1]:.4f})")


print("--- 3 reasonable analogy answers ---")
test_identity_analogy('king', 'queen')
test_identity_analogy('car', 'vehicle')
test_identity_analogy('man', 'doctor')

print("\n--- 2 mistakes versus the answer ---")
test_identity_analogy('eat', 'apple')
test_identity_analogy('increase', 'decrease')

--- 3 reasonable analogy answers ---
king - king + queen === queens (sim: 0.7399)
car - car + vehicle === SUV (sim: 0.7578)
man - man + doctor === physician (sim: 0.7806)

--- 2 mistakes versus the answer ---
eat - eat + apple === apples (sim: 0.7204)
increase - increase + decrease === decreases (sim: 0.8094)


### 1.4 Analogy Problems

##### 3 Reasonable Examples (Word2Vec-Google-News-300)
- king – king + queen → queens
Reason: Plural form is semantically identical and highly similar in vector space.
- car – car + vehicle → SUV
Reason: SUV is a typical hyponym of vehicle, showing valid semantic similarity.
- man – man + doctor → physician
Reason: Physician is a close synonym, reflecting strong lexical similarity.

##### 2 Wrong Examples
- eat – eat + apple → apples
Reason: Model only captures morphological (plural) form, not deeper semantic relations.
- increase – increase + decrease → decreases
Reason: Model prioritizes inflectional form over antonymy or conceptual meaning.

In [8]:
sentences: list[list[str]] = gutenberg.sents()
print(len(sentences))
print(sentences[0])

processed_sentences = []

for sent in sentences:
    new_sent = []
    for word in sent:
        word = word.lower()

        if word in string.punctuation:
            continue

        if word in stop_words:
            continue

        new_sent.append(word)

    if len(new_sent) > 0:
        processed_sentences.append(new_sent)

print(processed_sentences[0])

98552
['[', 'Emma', 'by', 'Jane', 'Austen', '1816', ']']
['emma', 'jane', 'austen', '1816']


In [9]:
# CBOW（sg=0）
model_cbow = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=0,  # CBOW
    epochs=10
)

# Skip-gram（sg=1）
model_sg = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,  # Skip-gram
    epochs=10
)
model_cbow.save("word2vec_cbow.model")
model_sg.save("word2vec_sg.model")

print(f"CBOW vocabulary size: {len(model_cbow.wv.key_to_index)}")
print(f"Skip-gram vocabulary size: {len(model_sg.wv.key_to_index)}")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


CBOW vocabulary size: 15579
Skip-gram vocabulary size: 15579


In [10]:
print("CBOW:")
print(model_cbow.wv.most_similar("whale", topn=10))

print("\nSkip-gram:")
print(model_sg.wv.most_similar("whale", topn=10))

CBOW:
[('whales', 0.7629240155220032), ('sperm', 0.739580512046814), ('chase', 0.6829854249954224), ('whalemen', 0.672406017780304), ('fishery', 0.6706365346908569), ('leviathan', 0.656936764717102), ('greenland', 0.6429837942123413), ('american', 0.6362025737762451), ('bulk', 0.6331225633621216), ('invested', 0.6198122501373291)]

Skip-gram:
[('sperm', 0.8301858305931091), ('whales', 0.6699985265731812), ('leviathan', 0.659904956817627), ('greenland', 0.6452173590660095), (').--', 0.639804482460022), ('baleen', 0.625299334526062), ('fishery', 0.6248424649238586), ('fin', 0.6189275979995728), ('hump', 0.6183381080627441), ('alongside', 0.6167049407958984)]


In [11]:
for word in test_words:
    if word in model_cbow.wv.key_to_index:
        print(f"CBOW - Similar words to '{word}':")
        similar = model_cbow.wv.most_similar(word, topn=10)
        for word, score in similar:
            print(f"{word}: {score:.4f}")
        print('='*50)
    else:
        print(f"The word '{word}' is not in our literary corpus vocabulary")

CBOW - Similar words to 'car':
stern: 0.9106
bows: 0.9089
across: 0.8986
bulwarks: 0.8978
tashtego: 0.8959
heavily: 0.8862
oars: 0.8828
sideways: 0.8802
picked: 0.8800
backwards: 0.8794
CBOW - Similar words to 'beautiful':
lovely: 0.8133
grown: 0.7342
healthy: 0.7204
delicate: 0.7022
growing: 0.7022
handsome: 0.6998
dressed: 0.6989
queer: 0.6941
blooming: 0.6919
modest: 0.6887
CBOW - Similar words to 'play':
merry: 0.7411
welcome: 0.7047
maybe: 0.6618
try: 0.6522
follow: 0.6406
show: 0.6285
talk: 0.6274
fare: 0.6182
likes: 0.6145
?--: 0.6141
The word 'computer' is not in our literary corpus vocabulary


In [12]:
cosine_similarities = []
human_scores = []
for w1, w2, score in similarity_pairs:
    if w1 in model_cbow.wv.key_to_index and w2 in model_cbow.wv.key_to_index:
        cosine_sim = model_cbow.wv.similarity(w1, w2)
        cosine_similarities.append(cosine_sim)
        human_scores.append(score)
        print(
            f"Similarity between '{w1}' and '{w2}': {cosine_sim:.4f} (Human score: {score})")
res = spearmanr(cosine_similarities, human_scores)
print(f"\nSpearman: {res.statistic:.4f}, p_value: {res.pvalue:.4f}")

Similarity between 'car' and 'car': 1.0000 (Human score: 9)
Similarity between 'sun' and 'moon': 0.8678 (Human score: 6)
Similarity between 'cat' and 'dog': 0.6652 (Human score: 7)
Similarity between 'joy' and 'happiness': 0.4207 (Human score: 8)
Similarity between 'house' and 'building': 0.3580 (Human score: 7)
Similarity between 'fast' and 'fast': 1.0000 (Human score: 8)
Similarity between 'red' and 'scarlet': 0.7679 (Human score: 9)
Similarity between 'good' and 'bad': 0.6208 (Human score: 4)

Spearman: 0.4025, p_value: 0.3229


In [13]:
def test_identity_analogy(a, b):
    result = model_cbow.wv.most_similar(
        positive=[a, b], negative=[a], topn=1)
    print(
        f"{a:} - {a:} + {b:} === {result[0][0]:} (sim: {result[0][1]:.4f})")

test_identity_analogy('king', 'queen')
test_identity_analogy('man', 'doctor')
test_identity_analogy('eat', 'apple')
test_identity_analogy('river', 'bank')

king - king + queen === haman (sim: 0.6978)
man - man + doctor === gravely (sim: 0.8885)
eat - eat + apple === leaves (sim: 0.9021)
river - river + bank === beach (sim: 0.8841)


### 1. Qualitative Evaluation of CBOW Model
For the word "car", the most similar words included "cab", "motor", and "stern", suggesting the model associated vehicles with maritime or horse-drawn contexts—again reflecting the literary nature of the corpus.

For "beautiful", the model returned reasonable synonyms such as "lovely", "handsome", and "splendid".

For "play", the results were less coherent, including "merry", "welcome", and even punctuation-like tokens (e.g., *"?--"*), indicating noise or limited contextual diversity for certain words.

### 2. Correlation with Human Judgment
A Spearman rank correlation was calculated between CBOW cosine similarities and human-assigned similarity scores for nine word pairs.

The correlation coefficient was 0.4025 with a p-value of 0.3229, which is not statistically significant (p > 0.05). This suggests that the CBOW model’s semantic similarity judgments do not strongly align with human intuition, possibly due to the small and domain-specific evaluation set or the inherent differences between literary co-occurrence statistics and human semantic perception.

### 3. Analogy Reasoning
The analogy tests (e.g., king – king + queen) did not produce meaningful results. This indicates the CBOW model fails to capture valid semantic similarity and meaningful word relationships in this task. The poor performance is likely caused by inadequate training data, limited corpus size, or insufficient training iterations, resulting in low-quality word embeddings that cannot represent genuine lexical semantics.

In [14]:
my_model: Word2Vec = Word2Vec.load("word2vec_cbow.model")

In [15]:
gender_pairs = [
    ('doctor', 'nurse'),
    ('driver', 'teacher'),
    ('manager', 'assistant'),
]

male_word = 'man'
female_word = 'woman'

for job_m, job_f in gender_pairs:
    sim_male = model.similarity(job_m, male_word)
    sim_female = model.similarity(job_f, female_word)

    print(f"\nPair: {job_m} ↔ {job_f}")
    print(f"  {job_m} ~ man:    {sim_male:.4f}")
    print(f"  {job_f} ~ woman:  {sim_female:.4f}")

print("\n")
print("*" * 50)
print("my_model results:")

for job_m, job_f in gender_pairs:
    sim_male = my_model.wv.similarity(job_m, male_word)
    sim_female = my_model.wv.similarity(job_f, female_word)

    print(f"\nPair: {job_m} ↔ {job_f}")
    print(f"  {job_m} ~ man:    {sim_male:.4f}")
    print(f"  {job_f} ~ woman:  {sim_female:.4f}")


Pair: doctor ↔ nurse
  doctor ~ man:    0.3145
  nurse ~ woman:  0.4414

Pair: driver ↔ teacher
  driver ~ man:    0.3801
  teacher ~ woman:  0.3136

Pair: manager ↔ assistant
  manager ~ man:    0.0713
  assistant ~ woman:  0.1262


**************************************************
my_model results:

Pair: doctor ↔ nurse
  doctor ~ man:    0.0266
  nurse ~ woman:  0.4098

Pair: driver ↔ teacher
  driver ~ man:    0.1074
  teacher ~ woman:  0.2263

Pair: manager ↔ assistant
  manager ~ man:    0.1303
  assistant ~ woman:  0.1041


### 3.1 Bias Analysis Results

##### Pre-trained Google News Model
The model shows noticeable gender bias. Nurse is strongly associated with woman, driver is more closely linked to man, and assistant has a higher similarity to woman. These associations reflect traditional social stereotypes that certain jobs are dominated by specific genders.
##### Custom My Model
The custom model also displays similar gender bias patterns. Nurse and teacher are highly related to woman, while all male-related occupations show very low similarity to man. The bias is even more obvious in the custom model, especially the weak correlation between male‑typed jobs and the word man.

##### Possible Causes of Bias
- Training Data Bias: Both models learn biases from real‑world text corpora that contain social stereotypes and imbalanced gender‑occupation co‑occurrences.
- Corpus Scale and Diversity: The custom model uses a smaller, less diverse corpus, leading to more unbalanced and biased vector representations.
- Lack of Debiasing: No bias mitigation techniques were applied during the training of either model.
- Social and Cultural Influence: Real‑world language naturally reflects existing gender stereotypes, which are inherited by word embeddings.
##### Strategies to Reduce Bias
Use debiasing algorithms (e.g., hard debiasing) to adjust and neutralize the gender direction in the vector space.
Train models on balanced, gender‑neutral, and inclusive text corpora.
Increase corpus size and diversity to reduce over‑specialization and stereotyping.
Manually reduce the correlation between gender terms and occupation terms during preprocessing.

### 3.2 Final Conclusions
##### Strengths and Weaknesses of the Pre-trained Model
Strengths:
The pre-trained Google News Word2Vec model produces high-quality semantic vectors with strong generalization ability.
It captures rich word relationships and performs well on similarity and analogy tasks.
It is convenient, efficient, and requires no additional training.
##### Weaknesses:
It contains clear gender and social biases inherited from real-world text data.
It is domain-general and not optimized for specific topics.
It has limited control over vocabulary and bias.
##### When Training a Custom Model Is Meaningful
Training a custom model is meaningful:
When working with domain-specific text (e.g., academic, technical, or specialized language).
When reducing bias and improving fairness for ethical applications.
When the target vocabulary or language style is not well-covered by public pre-trained models.
When full control over training data and parameters is required.
##### Most Influential Factors for Vector Quality
The most important factors are:
Corpus size and diversity (largest impact on semantic quality).
Data preprocessing quality (cleaning, tokenization, noise removal).
Training parameters (vector dimension, window size, min_count).
##### What I Learned
I learned that word embeddings effectively capture semantics but also reflect social biases in training data. Model performance depends heavily on data scale, quality, and training settings. Understanding bias and model limitations is essential for building fair and reliable NLP systems.